# Merging datasets

The Goal: Retrieve taxonomic datasets of multiple tomato rhizosphere studies

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder.
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [ ]:
# uncomment below if colab
#!pip install mgnipy

## Initiate client and prepare queries

In [1]:
from mgnipy import MGnipy

# init with a cache dir
MG = MGnipy(
    cache_dir="downloads"
)

# set up search params for studies resource
tomato_studies = MG.studies(
    biome_lineage="root:Host-associated:Plants:Rhizosphere", search="tomato"
)

# preview using helper method
tomato_studies.dry_run()

Planning the API call with params:
{'biome_lineage': 'root:Host-associated:Plants:Rhizosphere', 'search': 'tomato'}
Total requests to make: 1
Total records to retrieve: 17


## Execute queries

In [2]:
with MG: 
    # make the request 
    tomato_studies.get()

    # get detailed metadata for the studies
    tomato_studies.enrich_details()

# take a look at the study metadata
tomato_studies.metadata.to_pandas(expand_nested_dicts=True).head()

Enriching study details: 100%|██████████| 17/17 [00:01<00:00, 11.03it/s]


,accession,ena_accessions,title,updated_at,downloads,first_accession,metadata__study_name,metadata__center_name,metadata__study_title,metadata__study_accession,metadata__study_description,metadata__secondary_study_accession,biome__biome_name,biome__lineage
0,MGYS00006231,"[ERP139927, PRJEB55060]",EMG produced TPA metagenomics assembly of PRJN...,2026-10-31T19:05:01.728000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP139927,,EMG,EMG produced TPA metagenomics assembly of PRJN...,PRJEB55060,The Third Party Annotation (TPA) assembly was ...,ERP139927,Rhizosphere,root:Host-associated:Plants:Rhizosphere
1,MGYS00010296,"[ERP166137, PRJEB82448]",Microbiome-mediated tolerance of wild tomato t...,2026-05-28T15:46:48.615000+00:00,[],ERP166137,NaN,NaN,NaN,NaN,NaN,NaN,Rhizosphere,root:Host-associated:Plants:Rhizosphere
2,MGYS00010297,"[SRP333165, PRJNA755742]",Tomato rhizosphere microbiome in the pot exper...,2026-05-28T15:46:48.649000+00:00,[],SRP333165,NaN,NaN,NaN,NaN,NaN,NaN,Rhizosphere,root:Host-associated:Plants:Rhizosphere
3,MGYS00006204,"[ERP140102, PRJEB55219]",EMG produced TPA metagenomics assembly of PRJN...,2026-10-31T18:42:54.175000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP140102,,EMG,EMG produced TPA metagenomics assembly of PRJN...,PRJEB55219,The Third Party Annotation (TPA) assembly was ...,ERP140102,Rhizosphere,root:Host-associated:Plants:Rhizosphere
4,MGYS00006205,"[ERP140107, PRJEB55224]",EMG produced TPA metagenomics assembly of PRJN...,2026-10-31T18:43:03.297000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP140107,,EMG,EMG produced TPA metagenomics assembly of PRJN...,PRJEB55224,The Third Party Annotation (TPA) assembly was ...,ERP140107,Rhizosphere,root:Host-associated:Plants:Rhizosphere


## Access the `MGazine` of datasets

- study details have a `mgnipy.MGazine` which allow us to download and interact with study-level datasets outputed from MGnify.

- We can use `mgnipy.MGazine` to download the datasets onto disk or read them into our notebook.

- To access the study's mgazine use `.datasets`

- the __str__ representaiton of mgazine gives us a peak into the pipeline versions within, number of downloads and the short_description categories

In [3]:
# access study mgazine
MZ = tomato_studies.datasets

# filter downloads by short description 
ssu = MZ['Taxonomic assignments SSU']

# can check out downloads details as pandas df
ssu.downloads_df()

,file_type,download_type,short_description,long_description,alias,download_group,file_size_bytes,index_files,url,accession,pipeline_version
0,tsv,Taxonomic analysis,Taxonomic assignments SSU,Taxonomic assignments SSU (TSV),ERP139927_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00006231,v5
1,tsv,Taxonomic analysis,Taxonomic assignments SSU,Taxonomic assignments SSU (TSV),ERP140102_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00006204,v5
2,tsv,Taxonomic analysis,Taxonomic assignments SSU,Taxonomic assignments SSU (TSV),ERP140107_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00006205,v5
3,tsv,Taxonomic analysis,Taxonomic assignments SSU,Taxonomic assignments SSU (TSV),ERP139923_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00006230,v5
4,tsv,Taxonomic analysis,Taxonomic assignments SSU,Taxonomic assignments SSU (TSV),ERP140115_taxonomy_abundances_SSU_v5.0.tsv,study_summary.v5.0.taxonomic_analysis_ssu_rrna,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00006208,v5


Let's preview one of the files. You can read in whole or stream in chunks (via `chunksize`) a dataset by passing its `alias` or `url` to `MGazine.stream()`

In [4]:
# read in one of the taxonomic assignment datasets as a pandas df
df_example = ssu.stream(alias=ssu.aliases[0], chunksize=None, df_engine='pandas')

df_example.head()

,#SampleID,ERZ12343720,ERZ12343730,ERZ12343740,ERZ12343750,ERZ12343760,ERZ12343770,ERZ12343780,ERZ12343721,ERZ12343731,...,ERZ12590947,ERZ12593067,ERZ12590878,ERZ12590908,ERZ12590968,ERZ12590988,ERZ12590849,ERZ12590879,ERZ12590889,ERZ12590909
0,sk__Archaea,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,sk__Archaea;k__;p__Crenarchaeota,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,sk__Archaea;k__;p__Crenarchaeota;c__Thermoprot...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,sk__Archaea;k__;p__Thaumarchaeota;c__Nitrososp...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,sk__Archaea;k__;p__Thaumarchaeota;c__Nitrososp...,0,1,0,0,0,0,0,1,1,...,0,1,1,0,1,1,0,0,0,1


## Merging taxonomic datasets

For taxonomic datasets there is a special MGazine class, `TaxaMGazine`, that helps with merging the datasets and metadata. 

Then you can get the merged dataset as `to_pandas()` `to_polars()` `to_anndata()` for example. 

Otherwise you could manually read in the datasets and then merge them.

In [5]:
# accessing specialised TaxaMGazine
tmz = ssu.taxonomic

TaxaMGazine containing:
- MGnify pipeline versions: ['v5']
- Number of downloads: 5
- Short descriptions: ['Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_studies
-----------------------
Next steps: Use `.load()` to initialize.



In [6]:
# first lazy loading
tmz.load()

# accessing the 5 datasets in one df
tmz.to_anndata()

/Users/anglup/GitHub/mgnipy/.venv/lib/python3.13/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: X converted to numpy array with dtype float64
  return fn(*args_all, **kw)


AnnData object with n_obs × n_vars = 149 × 558
    var: 'Superkingdom', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species'

Note: 
- We can see that only the `var` or features (taxonomy) are annotated. The `obs` or observations (runs/samples) are not yet annotated because we did not collect their metadata. 
- From here we could use a `MGnetizer` to collect all the detailed metadata for the `.runs_accession` in our MGazine. 
- See the [Getting started -> Collecting more metadata] demo for more info. 

In [7]:
# tidying up cache
MG.clear_subcaches()